<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/4.1-training-foundations-ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CNN VS FINE-TUNING VS RANDOM FOREST


## Introduction

Using PyTorch and Torchvision, we will compare the performance of a pre-trained CNN, a fine-tuned model, and a Random Forest model for the image-classification task.

The input used to test our models will be a new audio file recorded by us. This clip will first be filtered, trimmed, and converted into a spectrogram. The parameters of that transformation must be strictly aligned with the input format we configured for our models.

To test the selected and saved model, we will provide a Web App where the user can upload their own audio clip; the system will automatically process it and show the prediction results for each model.

In [ ]:
# Importing necessary libraries

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import GradScaler, autocast # Para eficiencia en memoria
from google.colab import drive


In [ ]:
# Preprocessing and obtaining sets for training, test, validation.
#--------------------------------------------------------------------------------

# Connect to Google Drive

drive.mount('/content/drive')
ROOT_DIR = '/content/drive/MyDrive/ravdess_images_02'  # Ajustar a tu ruta :)
FAST_ROOT_DIR = '/content/features_local/ravdess_images_02'
# Device configuration (GPU if available, otherwise CPU)
# It is recommended to use the GPU provided by Google Colab to speed up training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Estas en modo: {device}")

# ImageFolder automatically assumes the classes are inside the root_dir folder
# In our case, these are the emotions.

def get_dataloaders(base_dir, feature_type,  batch_size=32):

  # Preprocessing: resize for ResNet, tensor conversion, normalization (ImageNet stats)
  data_transforms = transforms.Compose([
      #transforms.Resize((224, 224)), # If you exported the images with IMG_RES_02 = (224,224) in the previous notebook, comment out this line.
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalization for RGB channels required by ResNet
  ])


  # Assuming the structure is: ./dataset/features/mfcc/calm/01.png (standard RAVDESS structure, just like in the previous notebook)


  FEATURE_DIR = os.path.join(FAST_ROOT_DIR, feature_type)

  # Use PyTorch's native ImageFolder!
  full_dataset = datasets.ImageFolder(root=FEATURE_DIR, transform=data_transforms)

  # The class mapping is generated automatically thanks to PyTorch ImageFolder
  class_names = full_dataset.classes
  print(f"Emociones detectadas: {class_names}")
  print(f"Total images for class {feature_type}: {len(full_dataset)}")

  # 80-10-10 split: Train, Validation, Test respectively
  train_size = int(0.8 * len(full_dataset))
  val_size = int(0.1 * len(full_dataset))
  test_size = len(full_dataset) - train_size - val_size

  train_dataset, val_dataset, test_dataset = random_split(
      full_dataset, [train_size, val_size, test_size],
      generator=torch.Generator().manual_seed(42)
  )

  # DataLoaders: the most important sets
  batch_size = 32
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

  print("\n")
  print("The set sizes are:")
  print(f"Entrenamiento: {len(train_dataset)}")
  print(f"Validacion: {len(val_dataset)}")
  print(f"Test: {len(test_dataset)}")

  return train_loader, val_loader, test_loader, class_names

In [ ]:
# Copy the entire features folder from Drive to Colab's ultra-fast disk
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local

# Optional: If you have a .zip file in Drive, it is EVEN FASTER to copy the .zip and unzip it locally:
!cp /content/drive/MyDrive/ravdess_images_02.zip /content/
!unzip -q /content/ravdess_images_02.zip -d /content/features_local

In [ ]:
# Print root_dir content:
print(os.listdir(FAST_ROOT_DIR))

In [ ]:
get_dataloaders(FAST_ROOT_DIR, 'mel_spec', batch_size=32)

In [ ]:
# 3. MODEL CONSTRUCTION (FINE-TUNING)
# -------------------------------------------------------------------------------

def build_model(num_classes):
    """
    Carga ResNet18 preentrenado, congela las primeras capas y adapta la capa final.
    """
    # Load pre-trained ImageNet weights
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze parameters of the first layers,
    # since those layers already know how to detect basic edges and colors.
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last convolutional block (layer4) to learn
    # features specific to our spectrograms/MFCCs
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace the final fully connected layer (fc) so it
    # matches our number of emotions (e.g., 8 in RAVDESS)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model.to(device)

In [ ]:
from torch.overrides import TorchFunctionMode
from functools import total_ordering
# Training with an optimized cycle (this is where we actually use the GPU)
MODELS_DIR = '/content/drive/MyDrive/saved_models_ResNet'
def train_model(model, train_loader, val_loader, feature_type, epochs=15, lr=1e-4):  # Learnig Rate de 0.0001

  """
  Training loop; we use AMP (Automatic Mixed Precision) to speed up the process with 16-bit precision
  de esa manera se consume menos VRAM

  """

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

  # To avoid underflow in Automatic Mixed Precision (AMP).
  scaler = torch.amp.GradScaler('cuda')
  best_val_loss = float('inf')
  best_model_path = os.path.join(MODELS_DIR, f'best_model_for_{feature_type}')

  print(f"Starting training for feature: {feature_type}")

  for epoch in range(epochs):
      model.train() # Inicia el train del Modelo
      running_loss, correct, total = 0.0, 0 , 0

      for inputs, labels in train_loader:
          inputs, labels = inputs.to(device), labels.to(device)
          optimizer.zero_grad()
          # To dynamically cast tensors to FP16
          with torch.amp.autocast('cuda'):
              outputs = model(inputs)
              loss = criterion(outputs, labels)
          # Scaled backward pass to preserve precision
          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()

          # Training metric calculations
          running_loss += loss.item() * inputs.size(0)
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

      train_loss = running_loss / total
      train_acc = correct / total

      # Validation

      model.eval()
      val_loss, val_correct, val_total = 0.0, 0, 0

      # Disable gradients to save memory and time during validation
      with torch.no_grad():

        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

      val_loss = val_loss / val_total
      val_acc = val_correct / val_total

      print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

      # Save model checkpoints only if validation loss improves
      if val_loss < best_val_loss:
          best_val_loss = val_loss
          torch.save(model.state_dict(), best_model_path)

  print(f"Training completed. Best model saved as '{best_model_path}'\n")
  return best_model_path

In [ ]:
TARGET_FEATURE = 'delta2' # Here we define the feature used for training ej. 'mfcc', 'spec', 'mel_spec', etc.

# Load the data
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    base_dir=FAST_ROOT_DIR,
    feature_type=TARGET_FEATURE,
    batch_size=128
)

# Build the model dynamically:
model = build_model(num_classes=len(class_names))

# Run training:
best_model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    feature_type=TARGET_FEATURE,
    epochs=15,
    lr=1e-4
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, model_path, test_loader, class_names):
    """
    Load the best saved model and compute performance metrics on the test set.
    """
    #print(f"Loading the best model weights from: {model_path}")
    # Load the weights from the epoch with the lowest validation loss
    model.load_state_dict(torch.load(model_path))
    model.eval() # Evaluation mode: disables Dropout and Batch Normalization

    all_preds = []
    all_labels = []

    # Disable gradient computation to save memory (GPU allocation)
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            # Move tensors to the CPU so they can be used with Scikit-Learn
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 1. Classification report (F1-score, precision, recall, accuracy)
    print("\n" + "="*50)
    print("CLASSIFICATION REPORT (CONJUNTO DE PRUEBA)")
    print("="*50)
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Visual confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Model evaluation')
    plt.ylabel('Etiqueta Verdadera')
    plt.xlabel('Model prediction')
    plt.show()
    print(f"Matriz de confusion plana: \n {cm}")


In [ ]:

# Define the specific path you are going to evaluate now
current_path = '/content/drive/MyDrive/saved_models_ResNet/best_model_for_mfcc'
# Print outputs
print(f"\n{'='*60}")
print(f"Evaluating model: {os.path.basename(current_path)}")
print(f"{'='*60}\n")
# Call the evaluation function
evaluate_model(
    model=model,
    model_path=current_path,
    test_loader=test_loader,
    class_names=class_names
)


In [ ]:
current_path = '/content/drive/MyDrive/saved_models_ResNet/best_model_for_spec'

print(f"\n{'='*60}")
print(f"Evaluating model: {os.path.basename(current_path)}")
print(f"{'='*60}\n")

evaluate_model(
    model=model,
    model_path=current_path,
    test_loader=test_loader,
    class_names=class_names
)

In [ ]:
current_path = '/content/drive/MyDrive/saved_models_ResNet/best_model_for_mel_spec'

print(f"\n{'='*60}")
print(f"Evaluating model: {os.path.basename(current_path)}")
print(f"{'='*60}\n")

evaluate_model(
    model=model,
    model_path=current_path,
    test_loader=test_loader,
    class_names=class_names
)

In [ ]:
current_path = '/content/drive/MyDrive/saved_models_ResNet/best_model_for_delta'

print(f"\n{'='*60}")
print(f"Evaluating model: {os.path.basename(current_path)}")
print(f"{'='*60}\n")

evaluate_model(
    model=model,
    model_path=current_path,
    test_loader=test_loader,
    class_names=class_names
)

In [ ]:
current_path = '/content/drive/MyDrive/saved_models_ResNet/best_model_for_delta2'

print(f"\n{'='*60}")
print(f"Evaluating model: {os.path.basename(current_path)}")
print(f"{'='*60}\n")

evaluate_model(
    model=model,
    model_path=current_path,
    test_loader=test_loader,
    class_names=class_names
)